[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/value_extraction/value_extraction.ipynb)

# Pick a span the regex already found

Phones, amounts and dates have a shape a regular expression can list. Which of the listed spans is "the amount to refund" is a judgement. An LLM asked to copy the value often paraphrases it. Here the regex proposes the candidates, FRIDA-Decisions chooses one of them (or none), and code normalises the chosen span. The value that leaves the function is a substring of the text, not a generated one.

There is always a `none` option, so a list of spans is not forced into an answer. If the regex finds nothing, code returns `none` without calling the model.

Pattern: [Pre-parsed value extraction](https://docs.typesafe.ai/cookbooks/pre_parsed_value_extraction_cookbook). The numbers below are fake.


In [1]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


The regex owns the characters. The model only points at one of the spans it found, or at `none`.


In [3]:
import re

DATA = [
    {"kind": "money", "value": "500",
     "text": "К возврату 500 рублей. Доставка стоила 300 рублей и не возвращается."},
    {"kind": "money", "value": "2490",
     "text": "К возврату 2 490 рублей за сам заказ, доставка 300 рублей не возвращается."},
    {"kind": "phone", "value": "79160000000",
     "text": "Рабочий телефон 8 916 000-00-00. Старый номер 8 903 111-22-33 отключён."},
    {"kind": "phone", "value": "79990001122",
     "text": "Пишите только на +7 999 000-11-22. Номер 8 916 000-00-00 больше не обслуживается."},
    {"kind": "date", "value": "15 марта",
     "text": "Заказ оформили 2 марта, курьер привезёт его 15 марта."},
    {"kind": "date", "value": "3 апреля 2026",
     "text": "Оплатили 1 апреля 2026, доставка назначена на 3 апреля 2026."},
    {"kind": "money", "value": "none",
     "text": "Возврат оформили, точную сумму назовут после проверки склада."},
    {"kind": "phone", "value": "none",
     "text": "Перезвоните, когда будет минута, номер я пришлю отдельно."},
]
PATTERNS = {
    "phone": re.compile(r"(?:\+7|8)[\s\-]?\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}"),
    "money": re.compile(r"\d[\d\s]*\d\s*(?:руб(?:лей|ля|\.)?|₽)"),
    "date": re.compile(
        r"\d{1,2}\s+(?:января|февраля|марта|апреля|мая|июня|июля|августа|"
        r"сентября|октября|ноября|декабря)(?:\s+\d{4})?"
    ),
}
ASK = {
    "phone": "Какая из строк — телефон, на который просят писать?",
    "money": "Какая из строк — сумма к возврату?",
    "date": "Какая из строк — дата, когда привезут заказ?",
}


def spans(text, kind):
    found = []
    for match in PATTERNS[kind].finditer(text):
        span = match.group(0).strip()
        if span not in found:
            found.append(span)
    return found


def normalize(kind, span):
    if kind == "phone":
        digits = re.sub(r"\D", "", span)
        if digits.startswith("8"):
            return "7" + digits[1:]
        return digits
    if kind == "money":
        return str(int(re.sub(r"\D", "", span)))
    return re.sub(r"\s+", " ", span).strip()


def request_for(text, kind, found):
    criteria = {"none": "ни одна строка не отвечает на вопрос"}
    for i, span in enumerate(found):
        criteria[f"c{i}"] = span
    return {
        "state": text,
        "questions": {"pick": {"type": "choice", "instructions": ASK[kind], "criteria": criteria}},
    }


def extract(judge, text, kind):
    found = spans(text, kind)
    if not found:
        return {"value": "none", "confidence": None}
    answer = judge(request_for(text, kind, found))["answers"]["pick"]
    if answer["choice"] == "none":
        return {"value": "none", "confidence": answer["confidence"]}
    index = int(answer["choice"][1:])
    return {"value": normalize(kind, found[index]), "confidence": answer["confidence"]}


`value` is the normalised form of a span that occurs in the text, or `none`.


In [4]:
def run(judge):
    hits = 0
    for row in DATA:
        result = extract(judge, row["text"], row["kind"])
        if result["value"] == row["value"]:
            mark = "ok"
            hits += 1
        else:
            mark = "miss"
        if result["confidence"] is None:
            conf = "n/a"
        else:
            conf = f"{result['confidence']:.2f}"
        print(f"{mark:4} got {result['value']:16} gold {row['value']:16} p={conf}  {row['kind']}")
    print(f"\n{hits}/{len(DATA)} normalised values match")


In [5]:
run(judge)


ok   got 500              gold 500              p=0.71  money
ok   got 2490             gold 2490             p=0.94  money
ok   got 79160000000      gold 79160000000      p=0.97  phone
ok   got 79990001122      gold 79990001122      p=0.83  phone
ok   got 15 марта         gold 15 марта         p=0.42  date
ok   got 3 апреля 2026    gold 3 апреля 2026    p=0.43  date
ok   got none             gold none             p=n/a  money
ok   got none             gold none             p=n/a  phone

8/8 normalised values match
